# Memory

In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

In [19]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.language_models import FakeListChatModel
from langchain_core.callbacks.base import BaseCallbackHandler
from langchain_core.messages import trim_messages, HumanMessage


class PrintOutputCallback(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        print(f"Amount of input messages: {len(messages)}")


sessions = {}
handler = PrintOutputCallback()
llm = FakeListChatModel(responses=["ai1", "ai2", "ai3"])

def get_session_history(session_id: str):
    if session_id not in sessions:
        sessions[session_id] = InMemoryChatMessageHistory()
    return sessions[session_id]

trimmer = trim_messages(
    max_tokens=1,
    strategy="last",
    token_counter=len,
    include_system=True,
    start_on="human",
)

raw_chain = trimmer | llm
chain = RunnableWithMessageHistory(raw_chain, get_session_history)

config = {"callbacks": [PrintOutputCallback()], "configurable": {"session_id": "1"}}
_ = chain.invoke(
    [HumanMessage("Hi!")],
    config=config,
)
print(f"History length: {len(sessions['1'].messages)}")

_ = chain.invoke(
    [HumanMessage("How are you?")],
    config=config,
)
print(f"History length: {len(sessions['1'].messages)}")

Amount of input messages: 1
History length: 2
Amount of input messages: 1
History length: 4


In [3]:
sessions["1"].messages

[HumanMessage(content='Hi!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='ai1', additional_kwargs={}, response_metadata={}, id='lc_run--019c7f7b-f0ee-7ac2-990c-9901950fef73-0', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='How are you?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='ai2', additional_kwargs={}, response_metadata={}, id='lc_run--019c7f7b-f0ef-72e0-b85a-75bc8eb05c57-0', tool_calls=[], invalid_tool_calls=[])]

In [4]:
trimmer.invoke(sessions["1"].messages)

[]

In [5]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage
from langgraph.graph import START, END, StateGraph, MessagesState


def test_node(state: MessagesState):
    # ignore the last message since it's an input one
    messages = state["messages"]
    print(f"History length = {len(messages[:-1])}")
    return {"messages": [AIMessage(content="Hello!")]}


builder = StateGraph(MessagesState)
builder.add_node("test_node", test_node)
builder.add_edge(START, "test_node")
builder.add_edge("test_node", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [6]:
_ = graph.invoke({"messages": [HumanMessage(content="test")]}, config={"configurable": {"thread_id": "thread-a"}})
_ = graph.invoke({"messages": [HumanMessage(content="test")]}, config={"configurable": {"thread_id": "thread-b"}})
_ = graph.invoke({"messages": [HumanMessage(content="test")]}, config={"configurable": {"thread_id": "thread-a"}})

History length = 0
History length = 0
History length = 2


In [7]:
checkpoints = list(memory.list(config={"configurable": {"thread_id": "thread-a"}}))
for check_point in checkpoints:
  print(check_point.config["configurable"]["checkpoint_id"])

1f10f05e-d0bf-65d2-8004-6b99c446910e
1f10f05e-d0be-6dc6-8003-eb2df07457c1
1f10f05e-d0be-6650-8002-9468405c2c26
1f10f05e-d0bb-6112-8001-c4cec1b4cb82
1f10f05e-d0b8-6872-8000-6447e6967a37
1f10f05e-d0ae-6778-bfff-2e1ca4e979ae


In [8]:
checkpoint_id = checkpoints[-1].config["configurable"]["checkpoint_id"]
_ = graph.invoke(
    {"messages": [HumanMessage(content="test")]},
    config={"configurable": {"thread_id": "thread-a", "checkpoint_id": checkpoint_id}})

History length = 0


In [9]:
checkpoint_id = checkpoints[-3].config["configurable"]["checkpoint_id"]
_ = graph.invoke(
    {"messages": [HumanMessage(content="test")]},
    config={"configurable": {"thread_id": "thread-a", "checkpoint_id": checkpoint_id}})

History length = 2
